In [ ]:
!pip install mediapipe==0.10.14

In [ ]:
# define yoga poses to detect

num_classes = 5
class_labels = ['camel-pose', 'mountain-pose', 'triangle-pose', 'warrior-pose', 'wheel-pose']


In [ ]:
import cv2

In [ ]:
# import mediapipe library

import mediapipe as mp
mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils

In [ ]:
# pose detection model
pose = mp_pose.Pose()

In [ ]:
!unzip yoga.zip

In [ ]:
import os
import numpy as np

x = []
y = []

# build dataset by loading image files for each pose
for yoga_pose in class_labels:
  dir_path = os.path.join("yoga", yoga_pose)
  for item in os.listdir(dir_path):
    image_file = os.path.join(dir_path, item)
    if os.path.isfile(image_file):
      image = cv2.imread(image_file)

      if image is not None:

        # convert image to RGB
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # get landmarks
        output = pose.process(rgb_image)
        if output.pose_landmarks:
          y.append(yoga_pose)

          # extract co-ordinates for each landmark point
          landmarks = []
          for landmark in output.pose_landmarks.landmark:
            landmarks.append(landmark.x)
            landmarks.append(landmark.y)
            landmarks.append(landmark.z)

          for landmark in output.pose_world_landmarks.landmark:
            landmarks.append(landmark.x)
            landmarks.append(landmark.y)
            landmarks.append(landmark.z)

          x.append(landmarks)

# x stores landmark points
x = np.array(x)

# y stores corresponding pose name
y = np.array(y)

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [ ]:
x.shape

(204, 198)

In [ ]:
y.shape

(204,)

In [ ]:
# split data for training and testing

from sklearn.model_selection import train_test_split
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size=0.2)

In [ ]:
# choose and run ML model

from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier().fit(train_x, train_y)
model.score(test_x, test_y)

0.8780487804878049

In [ ]:
from sklearn.svm import SVC
model = SVC().fit(train_x, train_y)
model.score(test_x, test_y)

0.8048780487804879

In [ ]:
# define neural network architecture

import keras
from keras import layers

inputs = keras.Input(shape=(198, ))
dense1 = layers.Dense(32, activation="linear")(inputs)
dense2 = layers.Dense(32, activation="linear")(dense1)
dense3 = layers.Dense(32, activation="linear")(dense2)

outputs = layers.Dense(num_classes, activation="softmax")(dense3)
model = keras.Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 198)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         6,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,645 (33.77 KB)

 Trainable params: 8,645 (33.77 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

In [ ]:
# convert class labels from string to number

from sklearn.preprocessing import LabelEncoder
z = LabelEncoder().fit_transform(y)

In [ ]:
model.fit(x, z, validation_split=0.2, epochs=30, shuffle=True)